In [ ]:
# Run these commands in a separate cell first:
# !pip install yfinance openpyxl

"""
Quantitative Investment Workflow: Complete Analysis & Visualization
==================================================================
Full implementation with DCF valuation, factor modeling, MVO, and Monte Carlo stress testing
"""

import numpy as np
import pandas as pd
import yfinance as yf
from scipy.optimize import minimize
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

# Use a Colab-compatible style (fallback if needed)
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    plt.style.use('default')  # Colab fallback
sns.set_palette("husl")

# ============================================================================ #
# PART 1: DATA COLLECTION & FACTOR SCREENING
# ============================================================================ #

class FactorScreener:
    """AQR-style factor screening with comprehensive data storage"""

    def __init__(self, tickers, start_date='2019-01-01', end_date=None):
        self.tickers = tickers if isinstance(tickers, list) else [tickers]
        self.start_date = start_date
        self.end_date = end_date or datetime.now().strftime('%Y-%m-%d')
        self.price_data = None
        self.financial_data = None
        self.raw_data = {}

    def fetch_data(self):
        """Pull comprehensive price and fundamental data"""
        print(f"\n📊 Fetching data for {len(self.tickers)} tickers...")

        # Download price data
        try:
            price_data = yf.download(
                self.tickers,
                start=self.start_date,
                end=self.end_date,
                progress=False
            )

            if price_data.empty:
                raise ValueError("No price data downloaded")

            # Handle MultiIndex columns
            if isinstance(price_data.columns, pd.MultiIndex):
                adj_close = {}
                for ticker in self.tickers:
                    if ticker in price_data.columns.get_level_values(1):
                        adj_close[ticker] = price_data[('Adj Close', ticker)]
                self.price_data = pd.DataFrame(adj_close)
            else:
                self.price_data = pd.DataFrame({'Adj Close': price_data['Adj Close']})
                if len(self.tickers) == 1:
                    self.price_data.columns = self.tickers

            print(f"✅ Successfully fetched price data for {len(self.price_data.columns)} stocks")

        except Exception as e:
            print(f"❌ Error fetching price data: {e}")
            return None, None

        # Fetch comprehensive fundamental data
        financial_data = {}
        for ticker in self.tickers:
            try:
                stock = yf.Ticker(ticker)
                info = stock.info
                self.raw_data[ticker] = info

                financial_data[ticker] = {
                    'current_price': info.get('currentPrice', info.get('previousClose', np.nan)),
                    'market_cap': info.get('marketCap', np.nan),
                    'enterprise_value': info.get('enterpriseValue', np.nan),
                    'pe_ratio': info.get('trailingPE', np.nan),
                    'pb_ratio': info.get('priceToBook', np.nan),
                    'ps_ratio': info.get('priceToSalesTrailing12Months', np.nan),
                    'peg_ratio': info.get('pegRatio', np.nan),
                    'ev_ebitda': info.get('enterpriseToEbitda', np.nan),
                    'roe': info.get('returnOnEquity', np.nan),
                    'roa': info.get('returnOnAssets', np.nan),
                    'profit_margin': info.get('profitMargins', np.nan),
                    'operating_margin': info.get('operatingMargins', np.nan),
                    'debt_to_equity': info.get('debtToEquity', np.nan),
                    'current_ratio': info.get('currentRatio', np.nan),
                    'interest_coverage': info.get('interestCoverage', np.nan),
                    'revenue_growth': info.get('revenueGrowth', np.nan),
                    'earnings_growth': info.get('earningsGrowth', np.nan),
                    'beta': info.get('beta', 1.0),
                    'volatility_1y': info.get('volatility1Year', np.nan),
                    'dividend_yield': info.get('dividendYield', 0.0),
                    'payout_ratio': info.get('payoutRatio', np.nan),
                    'free_cash_flow': info.get('freeCashflow', np.nan),
                    'operating_cash_flow': info.get('operatingCashflow', np.nan),
                    'sector': info.get('sector', 'Unknown'),
                    'industry': info.get('industry', 'Unknown'),
                    'employees': info.get('fullTimeEmployees', np.nan)
                }
                print(f"✅ Fetched fundamentals for {ticker}")

            except Exception as e:
                print(f"⚠️  Could not fetch data for {ticker}: {e}")
                financial_data[ticker] = {k: np.nan for k in [
                    'current_price', 'market_cap', 'enterprise_value', 'pe_ratio', 'pb_ratio',
                    'ps_ratio', 'peg_ratio', 'ev_ebitda', 'roe', 'roa', 'profit_margin',
                    'operating_margin', 'debt_to_equity', 'current_ratio', 'interest_coverage',
                    'revenue_growth', 'earnings_growth', 'beta', 'volatility_1y', 'dividend_yield',
                    'payout_ratio', 'free_cash_flow', 'operating_cash_flow', 'sector', 'industry',
                    'employees'
                ]}

        self.financial_data = pd.DataFrame(financial_data).T
        return self.price_data, self.financial_data

    def calculate_factors(self):
        """Calculate AQR-style factor exposures with validation"""
        print("\n🎯 Calculating factor scores...")

        if self.price_data is None or self.price_data.empty:
            raise ValueError("No price data available for factor calculation")

        # Remove stocks with insufficient data
        valid_stocks = self.price_data.dropna(axis=1, thresh=252).columns
        if len(valid_stocks) == 0:
            raise ValueError("No stocks have sufficient price data")

        price_data_clean = self.price_data[valid_stocks]

        # 1. Momentum (12-1 month)
        returns = price_data_clean.pct_change()
        momentum = (price_data_clean / price_data_clean.shift(252) - 1) - (price_data_clean / price_data_clean.shift(21) - 1)
        momentum = momentum.iloc[-1]

        # 2. Quality (composite: ROE, profit margin, low debt)
        quality_scores = pd.Series(index=valid_stocks, dtype=float)
        if 'roe' in self.financial_data.columns and 'profit_margin' in self.financial_data.columns:
            for stock in valid_stocks:
                if stock in self.financial_data.index:
                    roe = self.financial_data.loc[stock, 'roe'] or 0
                    pm = self.financial_data.loc[stock, 'profit_margin'] or 0
                    quality_scores[stock] = (roe + pm) / 2

        # 3. Value (low P/E, low P/B)
        value_scores = pd.Series(index=valid_stocks, dtype=float)
        if 'pe_ratio' in self.financial_data.columns and 'pb_ratio' in self.financial_data.columns:
            for stock in valid_stocks:
                if stock in self.financial_data.index:
                    pe = self.financial_data.loc[stock, 'pe_ratio']
                    pb = self.financial_data.loc[stock, 'pb_ratio']
                    if pd.notna(pe) and pd.notna(pb):
                        value_scores[stock] = - (pe + pb)

        # 4. Low Volatility (12-month rolling std)
        low_vol_scores = -returns.rolling(252).std().iloc[-1]

        # 5. Size (small cap premium) - negative market cap
        size_scores = pd.Series(index=valid_stocks, dtype=float)
        if 'market_cap' in self.financial_data.columns:
            for stock in valid_stocks:
                if stock in self.financial_data.index:
                    mc = self.financial_data.loc[stock, 'market_cap']
                    size_scores[stock] = -mc if pd.notna(mc) else 0

        # Create factor DataFrame
        factors = pd.DataFrame({
            'momentum': momentum,
            'quality': quality_scores,
            'value': value_scores,
            'low_vol': low_vol_scores,
            'size': size_scores
        })

        # Rank and standardize (0 to 1)
        for col in factors.columns:
            factors[col] = factors[col].rank(pct=True, na_option='keep')

        # Composite factor score
        factors['composite_score'] = factors.mean(axis=1, skipna=True)
        factors = factors.sort_values('composite_score', ascending=False)

        print(f"✅ Calculated factor scores for {len(factors)} stocks")
        return factors


# ============================================================================ #
# PART 2: DCF VALUATION WITH DETAILED BREAKDOWN
# ============================================================================ #

class DCFValuator:
    """2-stage DCF model with detailed per-stock breakdown"""

    def __init__(self, tickers, financial_data, years_proj=5, perpetuity_rate=0.025):
        self.tickers = tickers
        self.financial_data = financial_data
        self.years_proj = years_proj
        self.perpetuity_rate = perpetuity_rate
        self.dcf_details = {}

    def compute_intrinsic_values(self):
        """Calculate intrinsic value for each stock with detailed breakdown"""
        print("\n💰 Running DCF valuations...")

        intrinsic_values = {}

        for ticker in self.tickers:
            try:
                if ticker not in self.financial_data.index:
                    continue

                # Get current financials
                current_fcf = self.financial_data.loc[ticker, 'free_cash_flow']
                revenue = self.financial_data.loc[ticker, 'operating_cash_flow'] or 1
                revenue_growth = self.financial_data.loc[ticker, 'revenue_growth'] or 0.05
                beta = self.financial_data.loc[ticker, 'beta'] or 1.0
                current_price = self.financial_data.loc[ticker, 'current_price']

                if pd.isna(current_fcf) or current_fcf <= 0:
                    print(f"⚠️  Skipping DCF for {ticker}: Invalid FCF")
                    continue

                # DCF Assumptions
                risk_free = 0.045
                market_premium = 0.06
                discount_rate = risk_free + beta * market_premium

                # Project future FCF
                projected_fcf = []
                projected_revenues = []
                for year in range(1, self.years_proj + 1):
                    proj_revenue = revenue * (1 + revenue_growth) ** year
                    fcf_margin = current_fcf / revenue * 0.7
                    proj_fcf = proj_revenue * fcf_margin

                    projected_revenues.append(proj_revenue)
                    projected_fcf.append(proj_fcf)

                # Terminal Value
                terminal_fcf = projected_fcf[-1] * (1 + self.perpetuity_rate)
                terminal_value = terminal_fcf / (discount_rate - self.perpetuity_rate)

                # Present Values
                pv_fcf = [fcf / (1 + discount_rate) ** year
                         for year, fcf in enumerate(projected_fcf, 1)]
                pv_terminal = terminal_value / (1 + discount_rate) ** self.years_proj

                total_pv = sum(pv_fcf) + pv_terminal
                shares_outstanding = 1e9
                intrinsic_value_per_share = total_pv / shares_outstanding

                # Store detailed breakdown
                self.dcf_details[ticker] = {
                    'intrinsic_value': max(intrinsic_value_per_share, 0),
                    'dcf_assumptions': {
                        'current_fcf': current_fcf,
                        'revenue_growth': revenue_growth,
                        'discount_rate': discount_rate,
                        'perpetuity_rate': self.perpetuity_rate,
                        'projected_fcf': projected_fcf,
                        'pv_fcf': pv_fcf,
                        'terminal_value': terminal_value,
                        'pv_terminal': pv_terminal,
                        'total_pv': total_pv
                    },
                    'margin_of_safety': (intrinsic_value_per_share - current_price) / intrinsic_value_per_share
                                        if intrinsic_value_per_share > 0 else np.nan
                }

                intrinsic_values[ticker] = intrinsic_value_per_share

            except Exception as e:
                print(f"❌ DCF failed for {ticker}: {str(e)}")

        print(f"✅ Completed DCF for {len(intrinsic_values)} stocks")
        return pd.DataFrame.from_dict(intrinsic_values, orient='index', columns=['intrinsic_value'])


# ============================================================================ #
# PART 3: RETURN PROJECTION WITH ATTRIBUTION
# ============================================================================ #

class ReturnProjector:
    """Project returns with factor attribution analysis"""

    def __init__(self, tickers, factor_screener, dcf_valuator):
        self.tickers = tickers
        self.factor_screener = factor_screener
        self.dcf_valuator = dcf_valuator

    def get_factor_premia(self):
        """Historical factor premia from academic literature"""
        return {
            'market': 0.06,
            'value': 0.03,
            'momentum': 0.04,
            'quality': 0.02,
            'low_vol': 0.02,
            'size': 0.01
        }

    def project_returns(self):
        """Project returns with detailed attribution"""
        print("\n📈 Projecting future returns...")

        factor_scores = self.factor_screener.calculate_factors()
        factor_premia = self.get_factor_premia()
        dcf_values = self.dcf_valuator.compute_intrinsic_values()

        rf = 0.045
        return_projections = {}
        attribution = {}

        for ticker in self.tickers:
            if ticker not in factor_scores.index or ticker not in dcf_values.index:
                continue

            # Factor-based return
            score = factor_scores.loc[ticker, 'composite_score']
            if pd.isna(score):
                continue

            # Attribution by factor
            factor_attribution = {}
            total_factor_return = rf

            for factor in ['value', 'momentum', 'quality', 'low_vol', 'size']:
                if factor in factor_scores.columns:
                    factor_score = factor_scores.loc[ticker, factor]
                    premium = factor_premia[factor]
                    contribution = (factor_score - 0.5) * premium * 0.2
                    factor_attribution[factor] = contribution
                    total_factor_return += contribution

            factor_attribution['market'] = rf + factor_premia['market']
            total_factor_return += factor_premia['market']

            # DCF-implied return
            current_price = self.factor_screener.financial_data.loc[ticker, 'current_price']
            intrinsic_value = dcf_values.loc[ticker, 'intrinsic_value']
            dcf_return = 0

            if pd.notna(current_price) and pd.notna(intrinsic_value) and current_price > 0:
                dcf_return = (intrinsic_value / current_price) ** (1/5) - 1

            # Blended final return
            final_return = 0.7 * total_factor_return + 0.3 * dcf_return

            # Store attribution
            attribution[ticker] = {
                'factor_return': total_factor_return,
                'dcf_return': dcf_return,
                'blended_return': final_return,
                'factor_attribution': factor_attribution
            }

            return_projections[ticker] = final_return

        self.attribution_df = pd.DataFrame(attribution).T
        print(f"✅ Projected returns for {len(return_projections)} stocks")
        return pd.Series(return_projections)


# ============================================================================ #
# PART 4: MEAN-VARIANCE OPTIMIZATION
# ============================================================================ #

class MeanVarianceOptimizer:
    """Markowitz portfolio optimization with comprehensive output"""

    def __init__(self, expected_returns, price_data, constraints=None):
        self.expected_returns = expected_returns
        self.price_data = price_data
        self.constraints = constraints or {}

    def optimize(self):
        """Run optimization and return comprehensive results"""
        print("\n📊 Running Mean-Variance Optimization...")

        valid_stocks = self.expected_returns.dropna().index.intersection(self.price_data.columns)
        expected_returns_clean = self.expected_returns[valid_stocks]
        price_data_clean = self.price_data[valid_stocks]

        if len(valid_stocks) < 3:
            print("⚠️  Need at least 3 stocks, using equal weights")
            weights = pd.Series(1/len(valid_stocks), index=valid_stocks)
            return {
                'weights': weights,
                'expected_return': expected_returns_clean.mean(),
                'volatility': 0.15,
                'sharpe_ratio': 0.5,
                'stock_contributions': pd.DataFrame(),
                'efficiency_ratio': 0
            }

        returns = price_data_clean.pct_change().dropna()
        cov_matrix = returns.cov() * 252

        num_assets = len(valid_stocks)
        bounds = self.constraints.get('bounds', [(0, 0.3)] * num_assets)

        def portfolio_performance(weights):
            p_return = np.sum(expected_returns_clean * weights)
            p_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
            return p_return, p_vol

        def negative_sharpe(weights):
            p_return, p_vol = portfolio_performance(weights)
            if p_vol < 1e-8:
                return 1e9
            return -(p_return - 0.045) / p_vol

        constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
        init_guess = np.array([1/num_assets] * num_assets)

        result = minimize(negative_sharpe, init_guess, method='SLSQP',
                         bounds=bounds, constraints=constraints)

        if result.success:
            optimal_weights = result.x
            portfolio_return, portfolio_vol = portfolio_performance(optimal_weights)
            sharpe_ratio = (portfolio_return - 0.045) / portfolio_vol

            # Calculate stock contributions
            weights_series = pd.Series(optimal_weights, index=valid_stocks)
            contributions = weights_series * expected_returns_clean

            results = {
                'weights': weights_series,
                'expected_return': portfolio_return,
                'volatility': portfolio_vol,
                'sharpe_ratio': sharpe_ratio,
                'stock_contributions': contributions.sort_values(ascending=False),
                'efficiency_ratio': portfolio_return / portfolio_vol if portfolio_vol > 0 else 0
            }

            print(f"✅ Optimal Portfolio - Expected Return: {portfolio_return:.2%}, Volatility: {portfolio_vol:.2%}")
            return results
        else:
            print("⚠️  Optimization failed, using equal weights")
            weights = pd.Series(1/num_assets, index=valid_stocks)
            return {
                'weights': weights,
                'expected_return': expected_returns_clean.mean(),
                'volatility': 0.15,
                'sharpe_ratio': 0.5,
                'stock_contributions': pd.DataFrame(),
                'efficiency_ratio': 0
            }


# ============================================================================ #
# PART 5: MONTE CARLO SIMULATION WITH VISUALIZATION
# ============================================================================ #

class MonteCarloSimulator:
    """Enhanced Monte Carlo with full path tracking and visualization"""

    def __init__(self, optimal_portfolio, price_data, n_simulations=25000, time_horizon=252*2):
        if optimal_portfolio is None:
            raise ValueError("No valid portfolio provided")

        self.weights = optimal_portfolio['weights']
        self.price_data = price_data
        self.n_simulations = n_simulations
        self.time_horizon = time_horizon
        self.all_paths = None

    def simulate_forward_paths(self):
        """Generate and store all simulation paths"""
        print(f"\n🎲 Running {self.n_simulations:,} Monte Carlo simulations...")

        valid_stocks = self.weights.index.intersection(self.price_data.columns)
        weights_clean = self.weights[valid_stocks] / self.weights[valid_stocks].sum()

        returns = self.price_data[valid_stocks].pct_change().dropna()

        mean_returns = returns.mean().values * 252
        cov_matrix = returns.cov().values * 252

        # Cholesky decomposition
        try:
            L = np.linalg.cholesky(cov_matrix + np.eye(len(cov_matrix)) * 1e-8)
        except:
            cov_matrix += np.eye(len(cov_matrix)) * 1e-6
            L = np.linalg.cholesky(cov_matrix)

        # Simulate paths
        dt = 1/252
        initial_value = 100
        self.all_paths = np.zeros((self.n_simulations, self.time_horizon + 1))
        self.all_paths[:, 0] = initial_value

        for sim in range(self.n_simulations):
            if sim % 5000 == 0 and sim > 0:
                print(f"  Progress: {sim:,}/{self.n_simulations:,} simulations")

            # Correlated random shocks
            random_shocks = np.random.normal(size=(self.time_horizon, len(valid_stocks)))
            correlated_returns = random_shocks @ L.T

            # Simulate portfolio path
            portfolio_value = initial_value
            stock_values = np.ones(len(valid_stocks)) * initial_value

            for t in range(self.time_horizon):
                daily_rets = mean_returns * dt + correlated_returns[t] * np.sqrt(dt)
                stock_values *= (1 + daily_rets)
                portfolio_value = np.sum(stock_values * weights_clean.values)
                self.all_paths[sim, t+1] = portfolio_value

        print(f"✅ Completed {self.n_simulations:,} simulations")
        return self.all_paths

    def calculate_metrics(self):
        """Calculate comprehensive risk metrics"""
        if self.all_paths is None:
            raise ValueError("Run simulate_forward_paths() first")

        final_returns = self.all_paths[:, -1] / self.all_paths[:, 0] - 1

        metrics = {
            'expected_return': np.mean(final_returns),
            'volatility': np.std(final_returns),
            'var_5': np.percentile(final_returns, 5),
            'var_1': np.percentile(final_returns, 1),
            'max_drawdown': np.min(final_returns),
            'prob_profit': np.mean(final_returns > 0),
            'prob_loss_10': np.mean(final_returns < -0.10),
            'prob_loss_20': np.mean(final_returns < -0.20),
            'skewness': stats.skew(final_returns),
            'kurtosis': stats.kurtosis(final_returns),
            'best_case': np.max(final_returns),
            'median_return': np.median(final_returns)
        }

        return metrics

    def plot_simulation_paths(self, n_paths_to_plot=100, save_path=None):
        """Plot Monte Carlo simulation paths"""
        if self.all_paths is None:
            raise ValueError("Run simulate_forward_paths() first")

        fig, ax = plt.subplots(figsize=(14, 8))

        # Plot subset of paths for visibility
        time_index = np.arange(self.time_horizon + 1) / 252

        # Plot individual paths with transparency
        for i in range(min(n_paths_to_plot, self.n_simulations)):
            ax.plot(time_index, self.all_paths[i, :],
                   alpha=0.3, linewidth=0.5, color='blue')

        # Plot median path
        median_path = np.median(self.all_paths, axis=0)
        ax.plot(time_index, median_path,
               linewidth=3, color='red', label='Median Path')

        # Plot confidence intervals
        p5_path = np.percentile(self.all_paths, 5, axis=0)
        p95_path = np.percentile(self.all_paths, 95, axis=0)
        ax.fill_between(time_index, p5_path, p95_path,
                       alpha=0.2, color='gray', label='90% Confidence Interval')

        ax.set_xlabel('Time (Years)', fontsize=12)
        ax.set_ylabel('Portfolio Value ($)', fontsize=12)
        ax.set_title(f'Monte Carlo Simulation Paths ({self.n_simulations:,} Scenarios)',
                    fontsize=14, fontweight='bold')
        ax.legend(loc='upper left')
        ax.grid(True, alpha=0.3)

        # Add metrics text box
        metrics = self.calculate_metrics()
        textstr = (f'Expected Return: {metrics["expected_return"]:.2%}\n'
                  f'Volatility: {metrics["volatility"]:.2%}\n'
                  f'5% VaR: {metrics["var_5"]:.2%}\n'
                  f'Best Case: {metrics["best_case"]:.2%}\n'
                  f'Prob(Profit): {metrics["prob_profit"]:.2%}')

        props = dict(boxstyle='round', facestyle='wheat', alpha=0.5)
        ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=10,
                verticalalignment='top', bbox=props)

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')

        return fig


# ============================================================================ #
# COMPREHENSIVE REPORTING DASHBOARD
# ============================================================================ #

def create_comprehensive_report(results, tickers, initial_capital=100000):
    """Generate complete analysis report with all details"""

    print("\n" + "="*80)
    print("📋 COMPREHENSIVE INVESTMENT ANALYSIS REPORT")
    print("="*80)

    # 1. PORTFOLIO SUMMARY
    print("\n" + "─"*80)
    print("🎯 PORTFOLIO OPTIMIZATION SUMMARY")
    print("─"*80)

    opt_portfolio = results['optimal_portfolio']
    stress_metrics = results['stress_metrics']

    summary_data = {
        'Metric': [
            'Expected Annual Return (MVO)',
            'Portfolio Volatility',
            'Sharpe Ratio',
            'Efficiency Ratio',
            'Monte Carlo Expected Return',
            'Monte Carlo Volatility',
            '5% Value at Risk (VaR)',
            '1% Value at Risk (VaR)',
            'Probability of Profit',
            'Probability of Loss >10%',
            'Probability of Loss >20%',
            'Best Case Scenario',
            'Worst Case Scenario'
        ],
        'Value': [
            f"{opt_portfolio['expected_return']:.2%}",
            f"{opt_portfolio['volatility']:.2%}",
            f"{opt_portfolio['sharpe_ratio']:.3f}",
            f"{opt_portfolio['efficiency_ratio']:.3f}",
            f"{stress_metrics['expected_return']:.2%}",
            f"{stress_metrics['volatility']:.2%}",
            f"{stress_metrics['var_5']:.2%}",
            f"{stress_metrics['var_1']:.2%}",
            f"{stress_metrics['prob_profit']:.2%}",
            f"{stress_metrics['prob_loss_10']:.2%}",
            f"{stress_metrics['prob_loss_20']:.2%}",
            f"{stress_metrics['best_case']:.2%}",
            f"{stress_metrics['max_drawdown']:.2%}"
        ]
    }

    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))

    # 2. OPTIMAL ALLOCATIONS
    print("\n" + "─"*80)
    print("💼 OPTIMAL PORTFOLIO ALLOCATIONS")
    print("─"*80)

    allocations = pd.DataFrame({
        'Stock': opt_portfolio['weights'].index,
        'Weight': [f"{w:.2%}" for w in opt_portfolio['weights'].values],
        'Position_Size': [f"${w*initial_capital:,.2f}" for w in opt_portfolio['weights'].values],
        'Expected_Return': [f"{results['expected_returns'][stock]:.2%}"
                          if stock in results['expected_returns'].index else "N/A"
                          for stock in opt_portfolio['weights'].index],
        'Contribution_to_Portfolio': [f"{opt_portfolio['stock_contributions'][stock]:.3%}"
                                     for stock in opt_portfolio['weights'].index]
    }).sort_values('Weight', ascending=False)

    print(allocations.to_string(index=False))

    # 3. STOCK-LEVEL DETAILED ANALYSIS
    print("\n" + "─"*80)
    print("📈 STOCK-LEVEL DETAILED ANALYSIS")
    print("─"*80)

    detailed_analysis = []
    for ticker in tickers:
        if ticker not in results['factor_scores'].index:
            continue

        factor_score = results['factor_scores'].loc[ticker, 'composite_score']
        expected_ret = results['expected_returns'].get(ticker, np.nan)

        # DCF details
        dcf_intrinsic = np.nan
        margin_of_safety = np.nan
        if hasattr(results['dcf_valuator'], 'dcf_details') and ticker in results['dcf_valuator'].dcf_details:
            dcf_intrinsic = results['dcf_valuator'].dcf_details[ticker]['intrinsic_value']
            margin_of_safety = results['dcf_valuator'].dcf_details[ticker].get('margin_of_safety', np.nan)

        # Current fundamentals
        fin_data = results['screener'].financial_data.loc[ticker]

        detailed_analysis.append({
            'Ticker': ticker,
            'Current_Price': f"${fin_data['current_price']:.2f}" if pd.notna(fin_data['current_price']) else "N/A",
            'Market_Cap': f"${fin_data['market_cap']:,.0f}" if pd.notna(fin_data['market_cap']) else "N/A",
            'Factor_Score': f"{factor_score:.3f}" if pd.notna(factor_score) else "N/A",
            'Expected_Return': f"{expected_ret:.2%}" if pd.notna(expected_ret) else "N/A",
            'DCF_Intrinsic_Value': f"${dcf_intrinsic:.2f}" if pd.notna(dcf_intrinsic) else "N/A",
            'Margin_of_Safety': f"{margin_of_safety:.1%}" if pd.notna(margin_of_safety) else "N/A",
            'P/E_Ratio': f"{fin_data['pe_ratio']:.1f}" if pd.notna(fin_data['pe_ratio']) else "N/A",
            'P/B_Ratio': f"{fin_data['pb_ratio']:.1f}" if pd.notna(fin_data['pb_ratio']) else "N/A",
            'ROE': f"{fin_data['roe']:.1%}" if pd.notna(fin_data['roe']) else "N/A",
            'Profit_Margin': f"{fin_data['profit_margin']:.1%}" if pd.notna(fin_data['profit_margin']) else "N/A",
            'Revenue_Growth': f"{fin_data['revenue_growth']:.1%}" if pd.notna(fin_data['revenue_growth']) else "N/A",
            'Beta': f"{fin_data['beta']:.2f}" if pd.notna(fin_data['beta']) else "N/A",
            'Sector': fin_data['sector']
        })

    detailed_df = pd.DataFrame(detailed_analysis)
    print(detailed_df.to_string(index=False))

    # 4. FACTOR ATTRIBUTION ANALYSIS
    print("\n" + "─"*80)
    print("🎯 FACTOR CONTRIBUTION ANALYSIS")
    print("─"*80)

    try:
        attribution_summary = results['projector'].attribution_df
        top_stocks = attribution_summary.head(10) if len(attribution_summary) > 10 else attribution_summary

        for stock in top_stocks.index:
            attr = attribution_summary.loc[stock]
            print(f"\n{stock}:")
            print(f"  Blended Expected Return: {attr['blended_return']:.2%}")
            print(f"  Factor Return: {attr['factor_return']:.2%}")
            print(f"  DCF Return: {attr['dcf_return']:.2%}")

            if 'factor_attribution' in attr:
                print(f"  Factor Breakdown:")
                for factor, contrib in attr['factor_attribution'].items():
                    print(f"    {factor}: {contrib:.2%}")
    except:
        print("Factor attribution data not available")

    # 5. SAVE TO EXCEL
    try:
        # In Colab, this will save to /content/ directory
        with pd.ExcelWriter('investment_analysis_report.xlsx') as writer:
            summary_df.to_excel(writer, sheet_name='Portfolio_Summary', index=False)
            allocations.to_excel(writer, sheet_name='Optimal_Allocations', index=False)
            detailed_df.to_excel(writer, sheet_name='Stock_Details', index=False)
            if 'projector' in results and hasattr(results['projector'], 'attribution_df'):
                results['projector'].attribution_df.to_excel(writer, sheet_name='Factor_Attribution')
            results['factor_scores'].to_excel(writer, sheet_name='Factor_Scores')

        print("\n✅ Report saved to 'investment_analysis_report.xlsx'")
        print("📁 In Colab: Check the 'Files' tab on the left to download the Excel file")
        print("📁 Local: The file is saved in your working directory")
    except Exception as e:
        print(f"\n⚠️  Could not save Excel report: {e}")


# ============================================================================ #
# MAIN EXECUTION
# ============================================================================ #

def run_complete_workflow(tickers, initial_capital=100000, plot_simulation=True):
    """Run the complete quantitative investment workflow"""

    print("\n" + "="*80)
    print("🚀 QUANTITATIVE INVESTMENT WORKFLOW")
    print("="*80)

    # Step 1: Factor Screening
    print("\n[Step 1/5] Data Collection & Factor Screening...")
    screener = FactorScreener(tickers)
    screener.fetch_data()

    if screener.price_data.empty or screener.financial_data.empty:
        print("❌ Failed to fetch sufficient data")
        return None

    factor_scores = screener.calculate_factors()

    # Step 2: DCF Valuation
    print("\n[Step 2/5] DCF Valuation...")
    dcf_valuator = DCFValuator(tickers, screener.financial_data)

    # Step 3: Return Projection
    print("\n[Step 3/5] Return Projection...")
    projector = ReturnProjector(tickers, screener, dcf_valuator)
    expected_returns = projector.project_returns()

    if expected_returns.empty:
        print("❌ No valid return projections")
        return None

    # Step 4: Mean-Variance Optimization
    print("\n[Step 4/5] Portfolio Optimization...")
    mvo = MeanVarianceOptimizer(expected_returns, screener.price_data)
    optimal_portfolio = mvo.optimize()

    if optimal_portfolio is None:
        print("❌ Optimization failed")
        return None

    # Step 5: Monte Carlo Simulation
    print("\n[Step 5/5] Monte Carlo Stress Testing...")
    simulator = MonteCarloSimulator(optimal_portfolio, screener.price_data)
    simulator.simulate_forward_paths()

    if plot_simulation:
        print("\n📊 Generating Monte Carlo visualization...")
        fig = simulator.plot_simulation_paths(save_path='monte_carlo_simulation.png')
        plt.show()

    stress_metrics = simulator.calculate_metrics()

    # Compile results
    results = {
        'screener': screener,
        'dcf_valuator': dcf_valuator,
        'projector': projector,
        'factor_scores': factor_scores,
        'expected_returns': expected_returns,
        'optimal_portfolio': optimal_portfolio,
        'stress_metrics': stress_metrics
    }

    # Generate comprehensive report
    create_comprehensive_report(results, tickers, initial_capital)

    return results


# ============================================================================ #
# HOW TO RUN IN GOOGLE COLAB
# ============================================================================ #
# 1. Paste this entire code into a cell and run it
# 2. In a NEW cell, run:
#
#    test_tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'JPM', 'V', 'JNJ', 'HD', 'PG', 'DIS']
#    results = run_complete_workflow(tickers=test_tickers, initial_capital=100000, plot_simulation=True)
#
# 3. Check the left sidebar "Files" tab to download:
#    - investment_analysis_report.xlsx (comprehensive Excel report)
#    - monte_carlo_simulation.png (chart image)
# ============================================================================ #

# Optional: Run directly from this cell for quick testing
if __name__ == "__main__":
    # Smaller universe for faster testing in Colab
    test_tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'JPM']

    print("\n🎯 Running comprehensive investment analysis...")
    print(f"📈 Analyzing {len(test_tickers)} stocks...")

    results = run_complete_workflow(
        tickers=test_tickers,
        initial_capital=100000,
        plot_simulation=True
    )

    if results is not None:
        print("\n" + "🎉 Analysis completed successfully!")
        print("📁 Check the 'Files' tab to download your reports")
    else:
        print("\n" + "❌ Analysis failed. Check errors above.")


🎯 Running comprehensive investment analysis...
📈 Analyzing 5 stocks...

🚀 QUANTITATIVE INVESTMENT WORKFLOW

[Step 1/5] Data Collection & Factor Screening...

📊 Fetching data for 5 tickers...
❌ Error fetching price data: ('Adj Close', 'AAPL')


AttributeError: 'NoneType' object has no attribute 'empty'

In [ ]:
!pip install yfinance openpyxl

In [ ]:
test_tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'JPM', 'V', 'JNJ', 'HD', 'PG', 'DIS']
results = run_complete_workflow(tickers=test_tickers, initial_capital=500000, plot_simulation=True)



🚀 QUANTITATIVE INVESTMENT WORKFLOW

[Step 1/5] Data Collection & Factor Screening...

📊 Fetching data for 10 tickers...
❌ Error fetching price data: ('Adj Close', 'AAPL')


AttributeError: 'NoneType' object has no attribute 'empty'